# Zi Music Studio v2 (free, Google Colab, engine branch)

Generates full-length music tracks (up to **10 minutes**, native) on a free Colab T4 and saves RouteNote-ready releases to Google Drive. No paid API, no card.

**What's new in v2 (dev-engine):**
- **Native long tracks:** the model supports 10-600 s directly. 5-6 min tracks render in one run; if a render comes back short, the engine extends it with a continuation segment crossfaded on the beat grid.
- **Quality presets:** Fast / Standard / High. High uses the `acestep-v15-turbo-shift3` model when available, 14 steps, and generates 2 seeds, then picks the better one with automatic QC (clipping, silence, duration, spectrum).
- **Mastering chain:** high-pass, gentle EQ, glue compressor, stereo width, two-pass loudness normalize to -14 LUFS / -1 dBTP, safety limiter. Exports WAV 16-bit 44.1 kHz + FLAC + MP3 320k + MP3 128k preview.
- **Storage routine (never deletes):** every finished track gets an MD5-verified FLAC master and a 64k Opus phone copy, then the source WAV moves to `Zi_Music/_archive_wav`. Restore any time: the FLAC decodes back to the identical WAV.
- **Batch queue with resume:** edit the JSON queue, run it, and re-run safely after a Colab disconnect (finished titles are skipped).
- **Error handling:** server health checks, auto-restart, per-track retries, batch report in Drive.

**Run order:** cell 1 (setup, 10-20 min first time) > cell 2 (engine) > cell 3 (one track) or 4 (batch queue).

Model: **ACE-Step 1.5** (MIT licence, commercial use allowed). Label every release as AI-generated.


In [ ]:
#@title 1) Setup (run once per session)
import os, subprocess, sys, time, shutil, urllib.request, json

# GPU check
try:
    out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
except Exception:
    out = ""
if not out:
    raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", out)

# Disk check (models + outputs need room)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 12:
    print("WARNING: low disk. Runtime > Disconnect and delete runtime, then start fresh if downloads fail.")

OUT_ROOT = '/content/drive/MyDrive/Zi_Music'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive could not be mounted (", e, "). Saving to the Colab session instead: /content/Zi_Music (download it from the Files panel, it is lost when the session ends).")
    OUT_ROOT = '/content/Zi_Music'
os.makedirs(OUT_ROOT, exist_ok=True)
LOCAL_TMP = '/content/zi_work'
os.makedirs(LOCAL_TMP, exist_ok=True)

REPO = '/content/ACE-Step-1.5'
subprocess.run([sys.executable,"-m","pip","install","-q","uv"],check=True)
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable),"uv")
print("uv:", UV)
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","--depth","1","https://github.com/ace-step/ACE-Step-1.5",REPO],check=True)
print("Installing dependencies (several minutes)...")
# --- T4 fix: run the lyric encoder in fp32 (fp16 overflows -> NaN with lyrics) ---
_p = REPO + "/acestep/core/generation/handler/init_service_loader.py"
_s = open(_p).read()
if "_zi_fp32_lyric" not in _s:
    import re as _re
    _helper = '''

def _zi_fp32_lyric(model):
    import torch
    try:
        enc = model.encoder.lyric_encoder
        if next(enc.parameters()).dtype != torch.float16:
            return
    except Exception:
        return
    orig = enc.forward
    def _up(x):
        return x.float() if torch.is_tensor(x) and x.is_floating_point() else x
    def fwd(*a, **k):
        a = [_up(x) for x in a]
        k = {n: _up(x) for n, x in k.items()}
        enc.float()
        try:
            out = orig(*a, **k)
        finally:
            enc.half()
        if torch.is_tensor(out):
            return out.half()
        if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
            out.last_hidden_state = out.last_hidden_state.half()
            return out
        if isinstance(out, tuple):
            return tuple(x.half() if torch.is_tensor(x) and x.is_floating_point() else x for x in out)
        return out
    enc.forward = fwd
'''
    _m = _re.search(r"^([ \t]+)self\.model\.eval\(\)\n", _s, _re.M)
    assert _m, "eval() anchor not found"
    _s = _s[:_m.end()] + _m.group(1) + "_zi_fp32_lyric(self.model)\n" + _s[_m.end():] + _helper
    open(_p, "w").write(_s)
    print("Applied T4 fp32 lyric-encoder fix.")

r = subprocess.run([UV,"sync"],cwd=REPO,capture_output=True,text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit("Dependency install failed (see above).")

def start_server():
    env = dict(os.environ,
        ACESTEP_API_HOST='127.0.0.1', ACESTEP_API_PORT='8001',
        ACESTEP_CONFIG_PATH='acestep-v15-turbo',
        ACESTEP_LM_MODEL_PATH='acestep-5Hz-lm-0.6B',
        ACESTEP_LM_BACKEND='pt', ACESTEP_INIT_LLM=os.environ.get('ACESTEP_INIT_LLM','true'),
        ACESTEP_USE_FLASH_ATTENTION='false', MPLBACKEND='Agg')
    log = open('/content/acestep.log','w')
    srv = subprocess.Popen([UV,"run","acestep-api"],cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT)
    return srv

BASE = 'http://127.0.0.1:8001'
SERVER = start_server()
print("Starting the music model and downloading weights (first time: 5-15 min)...")
ready = False
for i in range(240):
    if SERVER.poll() is not None:
        time.sleep(2)
        if os.path.exists('/content/acestep.log'):
            tail = open('/content/acestep.log').read()[-4000:]
        else:
            tail = "(no log)"
        print(tail)
        raise SystemExit("The music server exited during startup (log above). Send this log to your assistant.")
    try:
        with urllib.request.urlopen(BASE+'/health', timeout=5) as f:
            if f.status == 200: ready = True; break
    except Exception:
        pass
    time.sleep(5)
if not ready:
    print(open('/content/acestep.log').read()[-4000:])
    raise SystemExit("The music server did not start (log above). Send this log to your assistant.")

# Detect available DiT models (e.g. the shift3 quality variant)
MODELS = []
try:
    with urllib.request.urlopen(BASE+'/v1/models', timeout=15) as f:
        _j = json.loads(f.read())
    if isinstance(_j, dict):  # both shapes seen: {'data': {'models': [...]}} or a bare list
        _d = _j.get('data', _j)
        _lst = _d.get('models', []) if isinstance(_d, dict) else _d
    else:
        _lst = _j
    MODELS = [m.get('name') or m.get('id') or m.get('model') if isinstance(m, dict) else str(m) for m in _lst]
    MODELS = [m for m in MODELS if m]
except Exception as e:
    print("Could not list models:", e)
print("Models:", MODELS or "(unknown)")
print("Ready. Run cell 2.")

# --- Version sync check: compare this copy against the canonical GitHub notebook
LOCAL_VERSION = "2.0-dev-engine"  # keep in sync with ENGINE_VERSION in cell 2
CANONICAL_URL = "https://raw.githubusercontent.com/ira7al/zi/main/colab/Zi_Music_Studio.ipynb"
try:
    import re as _re
    _src = urllib.request.urlopen(CANONICAL_URL, timeout=15).read().decode("utf-8")
    _m = _re.search(r'ENGINE_VERSION\s*=\s*"([^"]+)"', _src)
    _remote = _m.group(1) if _m else None
    if _remote and _remote != LOCAL_VERSION:
        print(f"UPDATE AVAILABLE: GitHub has version {_remote}, this copy is {LOCAL_VERSION}.")
        print("Refresh this notebook from GitHub (colab/Zi_Music_Studio.ipynb) before long renders.")
    elif _remote:
        print(f"Version check OK: {LOCAL_VERSION} matches GitHub.")
    else:
        print("Version check: could not read remote version; continuing.")
except Exception as _e:
    print("Version check skipped (offline or GitHub unreachable):", _e)


## 2) Engine
Run once. Defines generation, QC, mastering, saving and the batch queue.

In [ ]:
#@title 2) Engine (run once, defines everything)
import json, re, os, time, csv, math, datetime, subprocess, urllib.request, urllib.error, random

ENGINE_VERSION = "2.0-dev-engine"

QUALITY = {
  "fast":    {"steps": 8,  "thinking": False, "candidates": 1, "prefer_shift3": False},
  "standard":{"steps": 8,  "thinking": True,  "candidates": 1, "prefer_shift3": True},
  "high":    {"steps": 14, "thinking": True,  "candidates": 2, "prefer_shift3": True},
}

def pick_model(quality="standard"):
    if QUALITY[quality]["prefer_shift3"] and "acestep-v15-turbo-shift3" in (MODELS or []):
        return "acestep-v15-turbo-shift3"
    return "acestep-v15-turbo"

def post(path, payload, timeout=120, tries=3):
    last = None
    for a in range(tries):
        try:
            req = urllib.request.Request(BASE+path, data=json.dumps(payload).encode(), headers={"Content-Type":"application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as f:
                return json.loads(f.read())
        except Exception as e:
            last = e; time.sleep(3*(a+1))
    raise RuntimeError(f"API call {path} failed after {tries} tries: {last}")

def unwrap(x):
    return x.get("data", x) if isinstance(x, dict) else x

def server_alive():
    try:
        with urllib.request.urlopen(BASE+'/health', timeout=5) as f:
            return f.status == 200
    except Exception:
        return False

def ensure_server():
    global SERVER
    if server_alive(): return True
    print("  ! music server is down, restarting it (weights are cached, a few minutes)...")
    try:
        SERVER = start_server()
    except Exception as e:
        print("  ! restart failed:", e); return False
    for i in range(120):
        if server_alive(): print("  server back up."); return True
        time.sleep(5)
    print(open('/content/acestep.log').read()[-2000:])
    return False

def wait_task(tid, max_min=45):
    for _i in range(max_min*6):
        time.sleep(10)
        res = unwrap(post("/query_result", {"task_id_list":[tid]}))[0]
        if res["status"] == 2:
            raise RuntimeError("Generation failed: " + str(res.get("result"))[:400])
        if res["status"] == 1:
            return json.loads(res["result"])
    raise RuntimeError("Timed out waiting for the model.")

def download_item(item, dest):
    urllib.request.urlretrieve(BASE + item["file"], dest)
    return dest

def gen_candidates(spec, quality="standard"):
    """Submit one generation task (batch_size = candidates) and download results."""
    q = QUALITY[quality]
    payload = {"audio_format":"flac", "batch_size": q["candidates"], "thinking": q["thinking"],
               "inference_steps": q["steps"], "model": pick_model(quality),
               "audio_duration": spec["seconds"], "use_random_seed": True,
               "vocal_language": spec.get("lang","en")}
    if spec.get("bpm"): payload["bpm"] = spec["bpm"]
    if spec.get("key"): payload["key_scale"] = spec["key"]
    if spec.get("instrumental"):
        payload.update({"prompt": spec["idea"], "lyrics":"[Instrumental]", "instrumental": True})
    else:
        payload.update({"sample_mode": True, "sample_query": spec["idea"]})
    if not ensure_server():
        raise RuntimeError("music server unavailable")
    tid = unwrap(post("/release_task", payload))["task_id"]
    items = wait_task(tid)
    paths = []
    ts = datetime.datetime.now().strftime("%H%M%S")
    for i, it in enumerate(items):
        p = f"{LOCAL_TMP}/cand_{ts}_{i}.flac"
        download_item(it, p)
        paths.append((p, it))
    return paths

def ffprobe_sec(path):
    r = subprocess.run(["ffprobe","-v","quiet","-show_entries","format=duration","-of","csv=p=0",path],capture_output=True,text=True)
    try: return float(r.stdout.strip())
    except Exception: return 0.0

def measure(path):
    """QC metrics via ffmpeg: loudness, true peak, clipping, silence, spectral flatness."""
    m = {"duration": ffprobe_sec(path)}
    r = subprocess.run(["ffmpeg","-hide_banner","-nostats","-i",path,"-af",
        "loudnorm=I=-14:TP=-1.0:LRA=11:print_format=json,astats=metadata=1:measure_overall=Peak_level+RMS_level+Flat_factor:measure_perchannel=none",
        "-f","null","-"],capture_output=True,text=True)
    txt = r.stderr
    jm = re.search(r"\{[^{}]*\"input_i\"[^{}]*\}", txt, re.S)
    if jm:
        try:
            lj = json.loads(jm.group(0))
            m["input_i"] = float(lj.get("input_i",-70)); m["input_tp"] = float(lj.get("input_tp",0))
            m["input_lra"] = float(lj.get("input_lra",0)); m["input_thresh"] = float(lj.get("input_thresh",-70))
        except Exception: pass
    pk = re.search(r"Peak level dB:\s*(-?[\d.]+|inf)", txt)
    fl = re.search(r"Flat factor:\s*(-?[\d.]+)", txt)
    if pk: m["peak_db"] = None if "inf" in pk.group(1) else float(pk.group(1))
    if fl: m["flat_factor"] = float(fl.group(1))
    r2 = subprocess.run(["ffmpeg","-hide_banner","-nostats","-i",path,"-af",
        "silencedetect=noise=-45dB:d=2.5","-f","null","-"],capture_output=True,text=True)
    sil = sum(float(x) for x in re.findall(r"silence_duration:\s*([\d.]+)", r2.stderr))
    m["silence_sec"] = round(sil,1)
    return m

def qc_score(m, target_sec):
    """Higher is better. Penalize short duration, clipping, silence, static noise."""
    s = 100.0
    if m["duration"] < 0.92*target_sec: s -= 60*(1 - m["duration"]/max(target_sec,1))
    tp = m.get("input_tp")
    if tp is not None and tp > -0.3: s -= 25  # digital clipping
    sil_ratio = m["silence_sec"]/max(m["duration"],1)
    s -= min(30, 120*sil_ratio)
    ff = m.get("flat_factor")
    if ff is not None and ff > 12: s -= 20  # flat spectrum = noise/artifact
    if m.get("input_lra",0) and m["input_lra"] < 2: s -= 8  # dead dynamics
    return round(s,1)

def pick_best(cands, target_sec):
    """Score every candidate, return (best_path, best_item, qc_list)."""
    scored = []
    for p, it in cands:
        m = measure(p)
        scored.append((qc_score(m, target_sec), p, it, m))
    scored.sort(key=lambda x: -x[0])
    qc_list = [{"file":os.path.basename(p), "score":sc, **{k:v for k,v in m.items()}} for sc,p,it,m in scored]
    return scored[0][1], scored[0][2], qc_list


# --- Mastering chain (free: ffmpeg only) -------------------------------------
def master_audio(raw, out_wav, target_i=-14.0, target_tp=-1.0):
    """Clean + glue + loudness-normalize. Returns mastering report.
    Chain: high-pass mud, gentle low-shelf cut, presence lift, bus compressor,
    slight stereo width, two-pass loudnorm (-14 LUFS, TP -1), safety limiter."""
    m1 = subprocess.run(["ffmpeg","-hide_banner","-nostats","-i",raw,"-af",
        f"loudnorm=I={target_i}:TP={target_tp}:LRA=11:print_format=json","-f","null","-"],
        capture_output=True,text=True).stderr
    jm = re.search(r"\{[^{}]*\"input_i\"[^{}]*\}", m1, re.S)
    if not jm: raise RuntimeError("loudness measurement failed")
    L = json.loads(jm.group(0))
    ln = (f"loudnorm=I={target_i}:TP={target_tp}:LRA=11:measured_I={L['input_i']}:measured_TP={L['input_tp']}"
          f":measured_LRA={L['input_lra']}:measured_thresh={L['input_thresh']}:offset={L['target_offset']}:linear=true")
    chain = ("highpass=f=25,"
             "lowshelf=g=-1.5:f=110,"
             "equalizer=f=7500:t=q:w=2:g=1.2,"
             "acompressor=threshold=-18dB:ratio=2.5:attack=8:release=140:makeup=1.5dB,"
             "extrastereo=m=1.08,"
             + ln + ",aresample=44100,alimiter=limit=0.891:level=false")
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",raw,"-af",chain,
                    "-ar","44100","-ac","2","-sample_fmt","s16",out_wav],check=True)
    m2 = subprocess.run(["ffmpeg","-hide_banner","-nostats","-i",out_wav,"-af",
        f"loudnorm=I={target_i}:TP={target_tp}:LRA=11:print_format=json","-f","null","-"],
        capture_output=True,text=True).stderr
    jm2 = re.search(r"\{[^{}]*\"input_i\"[^{}]*\}", m2, re.S)
    out_m = json.loads(jm2.group(0)) if jm2 else {}
    return {"chain":"hp25/lowshelf110/presence7k5/comp2.5/width1.08/loudnorm2pass/limiter",
            "in_i":L.get("input_i"),"in_tp":L.get("input_tp"),
            "out_i":out_m.get("input_i"),"out_tp":out_m.get("input_tp")}

def make_cover(path, palette, seed):
    import numpy as np
    from PIL import Image
    rng = np.random.default_rng(seed)
    N = 3000
    y, x = np.mgrid[0:N, 0:N].astype("float32") / N
    f = lambda: rng.uniform(1.5, 5.0)
    v = (np.sin(x*f()*6+y*f()*4+rng.uniform(0,6)) + np.sin(y*f()*7-x*f()*3+rng.uniform(0,6)) + np.sin((x+y)*f()*5+rng.uniform(0,6)))/3
    v = (v+1)/2
    c = np.array(palette, dtype="float32")
    t = v[...,None]*2
    lo = np.clip(t,0,1); hi = np.clip(t-1,0,1)
    img = c[0]*(1-lo) + c[1]*(lo*(1-hi)) + c[2]*hi
    Image.fromarray(np.clip(img,0,255).astype("uint8"),"RGB").save(path,"JPEG",quality=92)

PALETTES = [
 [(5,5,20),(70,0,140),(0,200,255)],   # neon techno
 [(10,10,30),(0,120,160),(255,120,40)],# dusk orange
 [(15,5,25),(90,20,120),(255,60,120)], # magenta club
 [(5,20,25),(0,140,120),(240,230,140)],# sea green
 [(20,10,10),(140,40,20),(255,180,60)],# ember
]

def slug(s):
    return re.sub(r"[^A-Za-z0-9]+","-",s).strip("-")[:50] or "track"

def auto_title(lyrics, g):
    for line in (lyrics or "").splitlines():
        line = line.strip()
        if line and not line.startswith("["):
            return " ".join(line.split()[:5]).strip(",.!?;:").title()
    return f"{g} Track {datetime.datetime.now().strftime('%H%M')}"

def save_release(raw_flac, item, spec, artist_name, writer_name, qc_list, master_rep, ts):
    """Master + export (WAV/FLAC/MP3-320/preview-128) + cover + metadata to Drive."""
    lyrics = item.get("lyrics","") or ""
    metas = item.get("metas",{}) or {}
    title = (spec.get("title") or "").strip() or auto_title(lyrics, spec["genre"])
    folder = f"{OUT_ROOT}/{ts}_{slug(title)}"
    os.makedirs(folder, exist_ok=True)
    base = f"{folder}/{slug(title)}"
    wav = base + ".wav"
    print("  mastering (2-pass loudness, EQ, glue compressor, limiter)...")
    rep = master_audio(raw_flac, wav)
    master_rep.update(rep)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",wav,"-sample_fmt","s16",base+".flac"],check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",wav,"-b:a","320k",base+"_320kbps.mp3"],check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",wav,"-b:a","128k",base+"_preview_128k.mp3"],check=True)
    make_cover(folder+"/cover_3000x3000.jpg", PALETTES[hash(title)%len(PALETTES)], int(time.time()))
    open(folder+"/lyrics.txt","w").write(lyrics if not spec.get("instrumental") else "[Instrumental]")
    meta = {"title":title,"artist":artist_name,"writer":writer_name,"genre":spec["genre"],
            "language":spec.get("lang","en"),"instrumental":bool(spec.get("instrumental")),
            "bpm":metas.get("bpm"),"key":metas.get("keyscale"),
            "target_duration_sec":spec["seconds"],"actual_duration_sec":round(ffprobe_sec(wav),1),
            "prompt":item.get("prompt"),"seed":item.get("seed_value"),"created":ts,
            "engine_version":ENGINE_VERSION,"quality":spec.get("quality","standard"),
            "model":pick_model(spec.get("quality","standard")),
            "qc_candidates":qc_list,"mastering":master_rep,
            "ai_generated":True,
            "models":{"music":item.get("dit_model"),"lm":item.get("lm_model")},
            "licence":"ACE-Step 1.5, MIT licence (commercial use allowed). https://github.com/ace-step/ACE-Step-1.5 and https://huggingface.co/ACE-Step/Ace-Step1.5"}
    json.dump(meta, open(folder+"/metadata.json","w"), indent=2, ensure_ascii=False)
    open(folder+"/routenote_form.txt","w").write(f"""ROUTENOTE RELEASE: copy these into the form (Distribution > Create New Release)
Release title (single = track title): {title}
Artist: {artist_name}
Writer(s) (composer, and lyricist if lyrics): {writer_name}
Primary genre: {spec['genre']}
Language: {spec.get('lang','en')}
Explicit: check the lyrics, tick Explicit if needed
C line / P line: {artist_name}
Label: {artist_name}
Track audio: {os.path.basename(base)}.flac   (44.1 kHz stereo 16-bit) or the _320kbps.mp3 (the .wav is archived under Zi_Music/_archive_wav)
Artwork: cover_3000x3000.jpg (3000x3000 RGB JPG, no text, no web addresses)
Contributor role: Producer = {artist_name}
Do NOT credit or name the AI company in the release (RouteNote rule).
If RouteNote moderation asks which AI tool you used, tell them: ACE-Step 1.5 (MIT licence)
  https://github.com/ace-step/ACE-Step-1.5
  https://huggingface.co/ACE-Step/Ace-Step1.5
Note: RouteNote cannot send AI-generated tracks to Content ID / Amazon / Korean stores.
Do not describe the track as imitating any real artist.
""")
    newfile = not os.path.exists(OUT_ROOT+"/releases.csv")
    with open(OUT_ROOT+"/releases.csv","a",newline="") as fh:
        cw = csv.writer(fh)
        if newfile: cw.writerow(["created","title","artist","genre","bpm","key","folder"])
        cw.writerow([ts,title,artist_name,spec["genre"],metas.get("bpm"),metas.get("keyscale"),folder])
    meta["storage"] = storage_routine(wav)   # last step: verified FLAC, opus64 copy, WAV -> _archive_wav (never delete)
    json.dump(meta, open(folder+"/metadata.json","w"), indent=2, ensure_ascii=False)
    return folder, base, lyrics, metas, title


# --- Storage routine (per Drive/Zi_Music/README_storage_routine.txt: never delete) ---
def _pcm_md5(path):
    r = subprocess.run(["ffmpeg","-v","error","-i",path,"-map","0:a","-f","md5","-"],
                       capture_output=True,text=True)
    return r.stdout.strip()

def storage_routine(wav_path):
    """After a track is finished: verified FLAC master, 64k Opus copy, WAV archived.
    Additive only. Nothing is deleted; on MD5 mismatch the WAV stays in place."""
    import shutil
    rep = {"verified_flac":None, "opus64":None, "wav_archived":None, "md5_match":False}
    flac_out = wav_path[:-4] + ".verified.flac"
    if not os.path.exists(flac_out):
        subprocess.run(["ffmpeg","-y","-v","error","-i",wav_path,"-compression_level","8",flac_out],check=True)
    a, b = _pcm_md5(wav_path), _pcm_md5(flac_out)
    rep["verified_flac"] = flac_out
    rep["md5_match"] = (a == b and a != "")
    if not rep["md5_match"]:
        print(f"  STORAGE WARNING: PCM MD5 mismatch for {os.path.basename(wav_path)}; WAV left in place")
        return rep
    opus_out = wav_path[:-4] + ".opus64.opus"
    if not os.path.exists(opus_out):
        subprocess.run(["ffmpeg","-y","-v","error","-i",flac_out,"-vn","-c:a","libopus",
                        "-b:a","64k","-vbr","on",opus_out],check=True)
    rep["opus64"] = opus_out
    rel = os.path.relpath(wav_path, OUT_ROOT)
    dst = os.path.join(OUT_ROOT, "_archive_wav", rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(wav_path, dst)
    rep["wav_archived"] = dst
    print(f"  storage: FLAC verified (md5 match), opus64 copy made, WAV archived to _archive_wav/{rel}")
    return rep


# --- Track pipeline with long-duration fallback ------------------------------
def crossfade_join(path_a, path_b, out_path, bpm=None, fade_sec=None):
    """Beat-aligned equal-power crossfade of two wavs/flacs."""
    if fade_sec is None:
        beat = 60.0/(bpm or 124)
        fade_sec = round(2*4*beat, 3)  # 2 bars
    d = ffprobe_sec(path_a)
    st = max(0.0, d - fade_sec)
    fc = (f"[0:a]atrim=0:{d},asetpts=PTS-STARTPTS[a];"
          f"[1:a]asetpts=PTS-STARTPTS[b];"
          f"[a][b]acrossfade=d={fade_sec}:c1=tri:c2=tri,aresample=44100")
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",path_a,"-i",path_b,
                    "-filter_complex",fc,out_path],check=True)

def extend_track(base_path, item, spec, quality):
    """Fallback when native long render comes back short: generate a continuation
    conditioned on the tail (reference audio = continuation task), crossfade on the beat grid."""
    cur = ffprobe_sec(base_path)
    need = spec["seconds"] - cur
    if need < 20: return base_path, []
    print(f"  native render is {cur:.0f}s, extending by {need+20:.0f}s (continuation + crossfade)...")
    tail = f"{LOCAL_TMP}/tail_{int(time.time())}.flac"
    subprocess.run(["ffmpeg","-y","-loglevel","error","-sseof","-30","-i",base_path,tail],check=True)
    q = QUALITY[quality]
    payload = {"audio_format":"flac","batch_size":1,"thinking":False,
               "inference_steps":q["steps"],"model":pick_model(quality),
               "audio_duration":min(600, need+25),"use_random_seed":True,
               "vocal_language":spec.get("lang","en"),
               "prompt":spec["idea"],"reference_audio_path":tail,
               "lyrics":"[Instrumental]" if spec.get("instrumental") else item.get("lyrics","") or ""}
    if spec.get("bpm"): payload["bpm"] = spec["bpm"]
    if spec.get("key"): payload["key_scale"] = spec["key"]
    tid = unwrap(post("/release_task", payload))["task_id"]
    cont = wait_task(tid)[0]
    cont_path = f"{LOCAL_TMP}/cont_{int(time.time())}.flac"
    download_item(cont, cont_path)
    joined = f"{LOCAL_TMP}/joined_{int(time.time())}.flac"
    crossfade_join(base_path, cont_path, joined, bpm=(item.get("metas",{}) or {}).get("bpm") or spec.get("bpm"))
    return joined, [{"method":"reference_continuation","added_sec":round(ffprobe_sec(joined)-cur,1)}]

def make_track(spec, artist_name, writer_name, ts=None):
    """One full track: candidates -> QC pick -> long-render fallback -> master -> save."""
    quality = spec.get("quality","standard")
    ts = ts or datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    print(f"> {spec.get('name') or spec.get('title') or spec['genre']} | {spec['seconds']}s | quality={quality}")
    last_err = None
    for attempt in range(2):
        try:
            cands = gen_candidates(spec, quality)
            best_path, best_item, qc_list = pick_best(cands, spec["seconds"])
            print(f"  picked {os.path.basename(best_path)} (score {qc_list[0]['score']}, {qc_list[0]['duration']:.0f}s)")
            segments = []
            if qc_list[0]["duration"] < 0.92*spec["seconds"]:
                best_path, segments = extend_track(best_path, best_item, spec, quality)
            folder, base, lyrics, metas, title = save_release(best_path, best_item, spec, artist_name, writer_name, qc_list, {"segments":segments}, ts)
            print(f"  saved: My Drive/Zi_Music/{os.path.basename(folder)}")
            return {"folder":folder,"base":base,"lyrics":lyrics,"metas":metas,"title":title,"qc":qc_list[0]}
        except Exception as e:
            last_err = e
            print(f"  ! attempt {attempt+1} failed: {str(e)[:300]}")
            time.sleep(10)
    raise RuntimeError(f"track failed after 2 attempts: {last_err}")

def run_queue(queue, artist_name, writer_name):
    """Batch queue with resume: skips tracks whose folder already exists in Drive."""
    done, failed, skipped = [], [], []
    existing = set(os.listdir(OUT_ROOT)) if os.path.isdir(OUT_ROOT) else set()
    for spec in queue:
        ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        probe_title = (spec.get("title") or spec.get("name") or spec["genre"])
        if any(("_"+slug(probe_title)) in e or e.endswith("_"+slug(probe_title)) for e in existing):
            print(f"- skip (already in Drive): {probe_title}")
            skipped.append(probe_title); continue
        try:
            r = make_track(spec, artist_name, writer_name, ts)
            done.append(r["title"])
        except Exception as e:
            print(f"  X failed: {probe_title}: {str(e)[:200]}")
            failed.append({"track":probe_title,"error":str(e)[:300]})
    rep = {"ts":datetime.datetime.now().isoformat(),"done":done,"failed":failed,"skipped":skipped}
    try:
        with open(OUT_ROOT+"/batch_report.json","a") as fh: fh.write(json.dumps(rep)+"\n")
    except Exception: pass
    print(f"\nBatch finished: {len(done)} done, {len(failed)} failed, {len(skipped)} skipped.")
    return rep


In [ ]:
#@title 3) Studio: one track, your choices
import ipywidgets as w
from IPython.display import display, clear_output, Audio, Image as IPImage

GENRES = {
 "Melodic Techno": "120-124 bpm melodic techno, tight punchy kick, deep sub bass, emotional minor chord progression on a soft pluck, lush atmospheric pads, hypnotic arpeggio, distant reverb vocal chops, long DJ-friendly intro and outro, bittersweet 4am mood",
 "Indie Dance": "118-124 bpm indie dance, live-feel bass guitar groove, four-on-the-floor kick, funky guitar chops, analog synth stabs, warm claps, playful sun-soaked mood",
 "Peak Techno": "128 bpm peak-time techno, hard punchy kick, driving offbeat bass, hypnotic acid sequence, sharp hats, white-noise risers, relentless peak-hour club mood",
 "Afro House": "121-122 bpm deep afro house, smooth deep kick, warm sub, layered djembe and shaker groove, soulful chord stabs, flute-like lead, ocean-breeze pads",
 "Progressive": "122-126 bpm progressive house-techno, rolling bassline, crisp claps, slowly opening filter arpeggio, epic layered pads, festival riser, sunrise euphoria",
 "Downtempo": "100-105 bpm downtempo indie electronic, soft round kick, warm electric bass, dreamy washed guitars, mellow keys, vinyl texture, hazy nostalgic calm",
}
g_genre = w.Dropdown(options=list(GENRES), value="Melodic Techno", description="Style")
g_idea  = w.Text(value="a late night drive along the Red Sea coast", description="Theme", layout=w.Layout(width="520px"))
g_dur   = w.IntSlider(value=330, min=60, max=600, step=30, description="Seconds")
g_qual  = w.Dropdown(options=[("High (2 seeds, 14 steps, best pick)","high"),("Standard (8 steps)","standard"),("Fast draft","fast")], value="high", description="Quality")
g_instr = w.Checkbox(value=True, description="Instrumental (no vocals)")
g_key   = w.Text(value="", placeholder="optional, e.g. Am", description="Key")
g_bpm   = w.IntText(value=0, description="BPM (0=auto)")
g_title = w.Text(value="", placeholder="empty = auto title", description="Title", layout=w.Layout(width="520px"))
g_artist= w.Text(value="Rahal", description="Artist", layout=w.Layout(width="520px"))
g_writer= w.Text(value="Rahal", description="Writer", layout=w.Layout(width="520px"))
g_btn   = w.Button(description="Generate", button_style="success", icon="music")
g_out   = w.Output()

def on_gen(_):
    with g_out:
        clear_output()
        try:
            spec = {"name": g_genre.value, "genre": g_genre.value, "seconds": g_dur.value,
                    "idea": GENRES[g_genre.value] + ". Theme: " + g_idea.value,
                    "instrumental": g_instr.value, "quality": g_qual.value, "lang": "en",
                    "title": g_title.value}
            if g_bpm.value: spec["bpm"] = g_bpm.value
            if g_key.value.strip(): spec["key"] = g_key.value.strip()
            print("Generating... a long high-quality track can take 10-25 min on a free T4. Do not close this tab.")
            r = make_track(spec, g_artist.value, g_writer.value)
            clear_output()
            print(f"Saved: My Drive/Zi_Music/{os.path.basename(r['folder'])}")
            print(f"Title: {r['title']} | BPM {r['metas'].get('bpm')} | key {r['metas'].get('keyscale')} | QC {r['qc']['score']}")
            display(Audio(r["base"]+"_preview_128k.mp3"))
            display(IPImage(r["folder"]+"/cover_3000x3000.jpg", width=220))
        except Exception as e:
            print("Error:", e)
            print(open('/content/acestep.log').read()[-1200:])

g_btn.on_click(on_gen)
display(w.VBox([g_genre, g_idea, g_dur, g_qual, g_instr, g_key, g_bpm, g_title, g_artist, g_writer, g_btn, g_out]))


In [ ]:
#@title 4) Batch queue: a week of tracks back to back (run cell 2 first)
import ipywidgets as w
from IPython.display import display, clear_output

DEFAULT_QUEUE = [
 {"name":"Melodic Techno 122 Night Drive","genre":"melodic techno","bpm":122,"seconds":330,"instrumental":False,"quality":"high",
  "idea":"122 BPM melodic techno, warm analog bassline, rolling kick, lush detuned supersaw pads, plucky arpeggiated lead, soft piano stabs. Hopeful late-night city drive. Structure: 32-bar intro, long build, main melodic drop, piano breakdown, final drop, DJ-friendly outro.",
  "lyric_note":"Short emotive lines about night drives and city lights, one repeating 2-line hook, open vowels."},
 {"name":"Melodic Techno 120 4am","genre":"melodic techno","bpm":120,"seconds":330,"instrumental":True,"quality":"high",
  "idea":"120 BPM melodic techno, tight kick, deep sub bass, emotional minor progression on soft pluck, string pad swells, gentle arpeggio, distant vocal chops as texture only. Introspective 4am feeling. Ambient pad intro, beat entry, melodic theme, long breakdown, emotional final drop, fade outro."},
 {"name":"Indie Dance 118 Kitchen Light","genre":"indie dance","bpm":118,"seconds":300,"instrumental":False,"quality":"standard",
  "idea":"118 BPM indie dance, live-feel bass groove, four-on-the-floor kick, funky guitar chops, analog brass stabs, warm claps. Playful, sun-soaked, slightly retro. Bass-only intro, verse groove, pre-chorus build, big chorus, instrumental bridge, double chorus, groove outro.",
  "lyric_note":"Conversational cheeky lyrics, everyday imagery, two short verses and a big repetitive chorus."},
 {"name":"Peak Techno 128 Warehouse","genre":"techno","bpm":128,"seconds":300,"instrumental":True,"quality":"standard",
  "idea":"128 BPM peak-time techno, hard punchy kick, driving offbeat bass, hypnotic acid sequence, sharp hats, white-noise risers. Relentless warehouse energy. 32-bar DJ intro, acid fade-in, first drop, riser breakdown with snare roll, bigger second drop, 32-bar beat outro."},
 {"name":"Afro House 121 Bonfire","genre":"afro house","bpm":121,"seconds":300,"instrumental":False,"quality":"standard",
  "idea":"121 BPM deep afro house, smooth deep kick, warm sub, layered djembe and shaker groove, soulful chord stabs, flute-like lead, ocean-breeze pads. Warm beach bonfire evening. Percussion intro building layer by layer, bass entry, vocal verse, flute hook, drums-only breakdown, full final section, soft outro.",
  "lyric_note":"Soulful and simple, theme of gratitude and the sea, one verse one hook, long held notes on the hook."},
 {"name":"Progressive 124 Sunrise","genre":"progressive house techno","bpm":124,"seconds":330,"instrumental":True,"quality":"high",
  "idea":"124 BPM progressive house-techno, rolling bassline, crisp claps, filter arpeggio opening slowly, epic layered pads, festival riser. Building anticipation to a sunrise moment. Long filtered intro, groove lock, 32-bar slow build, euphoric drop, second shorter build and drop, filtered outro."},
 {"name":"Downtempo 100 Haze","genre":"downtempo indie electronic","bpm":100,"seconds":300,"instrumental":False,"quality":"standard",
  "idea":"100 BPM downtempo indie electronic, soft round kick, warm electric bass, dreamy washed guitars, mellow Rhodes keys, gentle arpeggio, vinyl crackle. Hazy nostalgic afternoon. Keys intro, laid-back groove, vocal verse, dreamy chorus, guitar bridge, soft outro.",
  "lyric_note":"Hazy nostalgic lyrics about an old summer, half-sung half-spoken, lots of space."},
]

q_text = w.Textarea(value=json.dumps(DEFAULT_QUEUE, indent=1), description="Queue JSON", layout=w.Layout(width="700px", height="260px"))
q_artist = w.Text(value="Rahal", description="Artist", layout=w.Layout(width="520px"))
q_writer = w.Text(value="Rahal", description="Writer", layout=w.Layout(width="520px"))
q_btn = w.Button(description="Run the queue", button_style="warning", icon="rocket")
q_out = w.Output()
NOTE = ("One track per spec, in order. Already-finished titles are skipped (safe to re-run after a disconnect). "
        "A 7-track high-quality week can take 2-4 hours on a free T4; Colab free sessions may cut off, just re-run this cell and it resumes.")

def on_queue(_):
    with q_out:
        clear_output()
        try:
            queue = json.loads(q_text.value)
            assert isinstance(queue, list) and queue, "queue must be a non-empty JSON list"
            for s in queue:
                s.setdefault("seconds", 300); s.setdefault("quality", "standard"); s.setdefault("lang", "en")
                if s.get("lyric_note") and not s.get("instrumental"):
                    s["idea"] += " Lyrics: " + s["lyric_note"]
            print(NOTE)
            run_queue(queue, q_artist.value, q_writer.value)
        except Exception as e:
            print("Queue error:", e)

q_btn.on_click(on_queue)
display(w.VBox([q_text, q_artist, q_writer, q_btn, q_out]))


## Notes
- **Free limits:** Colab free GPU sessions close after a few hours or when idle. Re-run cell 1, then cell 2, then re-run the batch cell; the queue resumes where it stopped.
- **Long tracks:** up to 600 s native. On a free T4 a 330 s high-quality track takes roughly 10-25 min. If a render returns short audio, the engine auto-extends it with a continuation segment joined by a beat-aligned crossfade.
- **If generation stalls or quality drops:** set `ACESTEP_INIT_LLM=false` before cell 1 (caption/lyrics then come from the preset text only) or use the Fast quality preset.
- **Upload to RouteNote:** upload by hand from the Drive folder; each release folder has `routenote_form.txt` to copy from.
- **RouteNote and AI music:** accepted with conditions (no imitation of real artists, do not credit the AI company, no Content ID / Amazon / Korean stores, extra moderation time).
- **Licence:** ACE-Step 1.5 is MIT. Check the current model page before each release batch.
